# Coffee Roastery — Data Model Extension

Extend the **Core Data Model (CDM)** with two domain-specific types: an `Alarm` (a `CogniteActivity` with one extra property) and a `RoastProfile` (a pure descriptive type — a roasting recipe). Then bundle them into a **domain data model** and a **solution data model** to see the layered-architecture pattern Cognite recommends for production.

**Prerequisite:** [`minimal_knowledge_graph.ipynb`](minimal_knowledge_graph.ipynb) — the `issue_smoke_alarm` activity it creates is enriched in-place by this notebook.

## Why extend the CDM at all?

The CDM covers concepts every industrial customer needs (`CogniteAsset`, `CogniteActivity`, `CogniteFile`, …). What it deliberately **doesn't** cover is anything domain-specific: a roast profile, a well completion, a turbine blade serial number, a batch genealogy. You add those as **typed extensions on top of the CDM** — keeping the CDM properties for free, adding only what's specific to your industry.

Per the [Cognite best-practice guide](https://docs.cognite.com/cdf/dm/dm_guides/dm_designing_scalable_models), prefer **custom views that `implement` the relevant CDM views** rather than using CDM views directly. This lets you add domain-specific properties and relationships without creating dead ends in the graph. The pattern is:

```
Container (physical storage)
  ↓ defines properties + a `requires` constraint pointing at a CDM container
View (logical schema)
  ↓ implements a CDM view → inherits all its properties
  ↓ maps the new properties from the new container
Data model
  ↓ bundles views into a versioned, deployable schema
```

The `requires` constraint is **load-bearing for query performance** — it lets DMS pre-join the new container with the CDM container at query plan time. Without it, traversal queries that span both layers fall back to slow paths and time out at scale.

## Layered architecture: domain vs solution models

Cognite recommends a [layered data model architecture](https://docs.cognite.com/cdf/dm/dm_guides/dm_designing_scalable_models):

| Layer | Purpose | In this notebook |
|---|---|---|
| **Source** | Raw copy of data from a single source system. | (The main notebook simulates this by writing CDM instances directly.) |
| **Enterprise / Domain** | Integrates and contextualizes data from multiple sources — the single source of truth. Broad and complete. | `CoffeeRoasteryExtension` — custom views + all CDM views, so GraphQL and AI agents see the full schema. |
| **Solution** | Tailored, read-only views for a specific application. A curated subset of the domain model. | `RoastQualitySolution` — only `RoastProfile`, `Alarm`, `CogniteAsset`, `CogniteTimeSeries`. |

Solution models reference the enterprise layer — never each other. They exist so that an application team can move fast with a smaller, stable schema, while the broader domain model evolves independently.

## What we add

```
Alarm                 ── implements CogniteActivity
    + peakTemperatureCelsius : float64

RoastProfile          ── implements CogniteDescribable
    + targetTemperatureCelsius : float64
    + roastDegree              : text
    + roastDurationMinutes     : float64
```

The existing `issue_smoke_alarm` activity from the main notebook is **enriched in place** through the new `Alarm` view — same node, additional property, no duplication. Three roast profiles are written as new nodes.

> **Indexes** (B-tree on filterable scalars, inverted on list properties like `tags`) improve query performance but aren't required for the pattern to work — add them on the container as `indexes={...}` once you know your access patterns.

## Setup

> **Env vars are loaded once per kernel.** `get_client()` reads `.env` via `load_dotenv()` the first time it runs in this kernel session. If you edit `.env` after the kernel has started, restart the kernel before re-running.

In [ ]:
import sys
from pathlib import Path

from cognite.client.data_classes.data_modeling import (
    ContainerApply,
    ContainerId,
    DataModelApply,
    MappedPropertyApply,
    NodeApply,
    NodeId,
    NodeOrEdgeData,
    ViewApply,
    ViewId,
)
from cognite.client.data_classes.data_modeling.containers import (
    ContainerPropertyApply,
    RequiresConstraintApply,
)
from cognite.client.data_classes.data_modeling.data_types import Float64, Text

# Works whether the kernel starts in the notebook folder or the repo root.
repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "helpers").is_dir()
)
sys.path.insert(0, str(repo_root))
from helpers import CDM_SPACE, CDM_VERSION, SPACE, get_client  # noqa: E402

In [ ]:
client = get_client(client_name="coffee-roastery-dm-extension")

## Schema — containers, views, data model

Three writes, in dependency order:

1. **Containers** — physical storage. Properties + `requires` constraint pointing at a CDM container.
2. **Views** — logical schema. `implements` a CDM view (so all CDM properties are inherited automatically); then maps each new container property as a view property.
3. **Data model** — versioned bundle of views, including the CDM views you depend on, so consumers see one self-contained schema.

In [ ]:
# 1) Containers. The `requires` constraint is what enables DMS to optimise
# cross-container joins — every node in this container must also exist in the
# referenced CDM container.

alarm_container = ContainerApply(
    space=SPACE,
    external_id="Alarm",
    name="Alarm",
    description="Extends CogniteActivity with alarm-specific properties",
    used_for="node",
    properties={
        "peakTemperatureCelsius": ContainerPropertyApply(
            type=Float64(),
            nullable=True,
            name="Peak Temperature (°C)",
            description="Highest process temperature recorded when the alarm triggered.",
        ),
    },
    constraints={
        "requires_activity": RequiresConstraintApply(
            require=ContainerId(CDM_SPACE, "CogniteActivity"),
        ),
    },
)

roast_profile_container = ContainerApply(
    space=SPACE,
    external_id="RoastProfile",
    name="Roast Profile",
    description="Roasting recipe — target conditions for a batch",
    used_for="node",
    properties={
        "targetTemperatureCelsius": ContainerPropertyApply(
            type=Float64(),
            nullable=False,
            name="Target Temperature (°C)",
            description="Peak drum temperature the profile aims for.",
        ),
        "roastDegree": ContainerPropertyApply(
            type=Text(),
            nullable=False,
            name="Roast Degree",
            description="Roast level descriptor: light, medium, dark, or french.",
        ),
        "roastDurationMinutes": ContainerPropertyApply(
            type=Float64(),
            nullable=True,
            name="Roast Duration (minutes)",
            description="Time in drum from charge to drop.",
        ),
    },
    constraints={
        "requires_describable": RequiresConstraintApply(
            require=ContainerId(CDM_SPACE, "CogniteDescribable"),
        ),
    },
)

containers = client.data_modeling.containers.apply(
    [alarm_container, roast_profile_container]
)
for c in containers:
    print(
        f"Container: {c.space}/{c.external_id}  props={list(c.properties.keys())}  constraints={list(c.constraints.keys())}"
    )

In [ ]:
# 2) Views. `implements` inherits all properties from the CDM view. Only the
# new properties need explicit mappings — CDM properties (name, tags, assets,
# startTime, …) are resolved automatically from the CDM containers.

alarm_view = ViewApply(
    space=SPACE,
    external_id="Alarm",
    name="Alarm",
    description="Safety alarm event. Extends CogniteActivity with peak process temperature.",
    version="v1",
    implements=[ViewId(CDM_SPACE, "CogniteActivity", CDM_VERSION)],
    properties={
        "peakTemperatureCelsius": MappedPropertyApply(
            container=ContainerId(SPACE, "Alarm"),
            container_property_identifier="peakTemperatureCelsius",
        ),
    },
)

roast_profile_view = ViewApply(
    space=SPACE,
    external_id="RoastProfile",
    name="Roast Profile",
    description="Roasting recipe — target temperature, degree, and duration for a batch.",
    version="v1",
    implements=[ViewId(CDM_SPACE, "CogniteDescribable", CDM_VERSION)],
    properties={
        "targetTemperatureCelsius": MappedPropertyApply(
            container=ContainerId(SPACE, "RoastProfile"),
            container_property_identifier="targetTemperatureCelsius",
        ),
        "roastDegree": MappedPropertyApply(
            container=ContainerId(SPACE, "RoastProfile"),
            container_property_identifier="roastDegree",
        ),
        "roastDurationMinutes": MappedPropertyApply(
            container=ContainerId(SPACE, "RoastProfile"),
            container_property_identifier="roastDurationMinutes",
        ),
    },
)

views = client.data_modeling.views.apply([alarm_view, roast_profile_view])
for v in views:
    impl = [f"{i.space}/{i.external_id}" for i in v.implements]
    print(
        f"View: {v.space}/{v.external_id}  v{v.version}  implements={impl}  total_props={len(v.properties)}"
    )

### Enterprise / domain data model

The next cell creates the **enterprise (domain) data model**: `CoffeeRoasteryExtension`. It bundles the custom views alongside all CDM views so that GraphQL, AI agents, and downstream applications see one complete, self-contained schema.

Including all of CDM is the right choice at the domain level — this is the single source of truth. Applications that only need a subset use a **solution data model** (shown after the instances section).

In [ ]:
# 3a) Enterprise / domain data model — broad and complete.
# Bundles every CDM view alongside the custom views so consumers see
# one self-contained schema (GraphQL, AI agents, dashboards).

cdm_views = client.data_modeling.views.list(
    space=CDM_SPACE, include_global=True, limit=-1
)

domain_model = DataModelApply(
    space=SPACE,
    external_id="CoffeeRoasteryExtension",
    name="Coffee Roastery Extension",
    version="v1",
    description="Enterprise/domain data model: custom roastery views + complete CDM.",
    views=[
        *cdm_views.as_ids(),
        ViewId(SPACE, "Alarm", "v1"),
        ViewId(SPACE, "RoastProfile", "v1"),
    ],
)

dm = client.data_modeling.data_models.apply(domain_model)
print(
    f"Domain model: {dm.space}/{dm.external_id}  v{dm.version}  ({len(dm.views)} views total)"
)
print()
print("Custom views:")
for v in dm.views:
    if v.space != CDM_SPACE:
        print(f"  {v.space}/{v.external_id}  v{v.version}")

### Solution data model

A **solution model** is a curated subset of the domain model, tailored for one application or use case. It references the same containers and views — no data duplication — but exposes only the views that application needs.

Here we create `RoastQualitySolution`: a hypothetical roast-quality dashboard that only cares about profiles, alarms, assets, and time series. It deliberately omits files, annotations, 3D, and most other CDM views. This keeps the schema small, stable, and fast to query.

> **Best practice ([Cognite docs](https://docs.cognite.com/cdf/dm/dm_guides/dm_designing_scalable_models)):** solution models should reference only the enterprise/domain layer — never each other. Map data directly from enterprise containers rather than duplicating it.

In [ ]:
# 3b) Solution data model — curated subset for a specific application.
# Only the views the roast-quality dashboard needs: profiles, alarms,
# assets, and time series. Everything else is left out.

solution_model = DataModelApply(
    space=SPACE,
    external_id="RoastQualitySolution",
    name="Roast Quality Solution",
    version="v1",
    description="Solution model for the roast-quality dashboard: profiles, alarms, assets, time series.",
    views=[
        ViewId(SPACE, "RoastProfile", "v1"),
        ViewId(SPACE, "Alarm", "v1"),
        ViewId(CDM_SPACE, "CogniteAsset", CDM_VERSION),
        ViewId(CDM_SPACE, "CogniteTimeSeries", CDM_VERSION),
    ],
)

sm = client.data_modeling.data_models.apply(solution_model)
print(
    f"Solution model: {sm.space}/{sm.external_id}  v{sm.version}  ({len(sm.views)} views)"
)
for v in sm.views:
    print(f"  {v.space}/{v.external_id}  version: {v.version}")

### ✅ Verify in Fusion

Open [Fusion](https://fusion.cognite.com) → **Search**. Click the **settings (gear) icon** in the top right of the page, and under **Data model**, select `CoffeeRoasteryExtension` (or `RoastQualitySolution`). `Alarm` and `RoastProfile` should now appear as searchable types alongside the standard CDM types.

> **Heads-up: `Alarm` will show 0 results here, and that's expected.** By default, a view only matches an instance once it has data in *every* container the view maps to. `Alarm` maps to both `CogniteActivity` and the new `Alarm` container, and right now `issue_smoke_alarm` only has data in `CogniteActivity` — nothing in `Alarm` yet. Once the **Instances** section below writes `peakTemperatureCelsius` through the `Alarm` view, it has data in both containers and shows up immediately.

## Instances

Two writes:

1. **Enrich `issue_smoke_alarm` in place** — the activity already exists from the main notebook. Writing it again through the `Alarm` view adds the new `peakTemperatureCelsius` property to the `Alarm` container, leaving the CDM properties (name, tags, assets, startTime, …) untouched. Same node, two containers, seen as one instance through the view.
2. **Write three new `RoastProfile` nodes** — light, medium, dark recipes. These are pure new instances (no pre-existing CDM node).

> **`replace=False` (patch mode)** — only the supplied properties are written. This is the safe default for incremental enrichment. `replace=True` would overwrite the entire node, dropping anything not in the request.

In [ ]:
# Enrich issue_smoke_alarm: add peakTemperatureCelsius via the Alarm view.
alarm_node = NodeApply(
    space=SPACE,
    external_id="issue_smoke_alarm",
    sources=[
        NodeOrEdgeData(
            source=ViewId(SPACE, "Alarm", "v1"),
            properties={"peakTemperatureCelsius": 260.0},
        )
    ],
)

result = client.data_modeling.instances.apply(nodes=[alarm_node], replace=False)
n = result.nodes[0]
print(f"Upserted: {n.space}/{n.external_id}  version={n.version}")

### ✅ Verify in Fusion

Search for `Smoke Alarm` in Fusion and open the activity. Under the **Properties** panel, switch to the `Alarm` view — you should see `peakTemperatureCelsius = 260.0` alongside the original CDM properties (name, tags, assets, start/end times). Same node, two containers.

If you don't see the `Alarm` view listed, try refreshing the page.

In [ ]:
# Three roast profiles. Note that `name`, `description`, and `tags` are
# CogniteDescribable properties — inherited automatically because RoastProfile
# implements CogniteDescribable. We write them all in a single API call.
roast_profile_view_id = ViewId(SPACE, "RoastProfile", "v1")

profiles = [
    NodeApply(
        space=SPACE,
        external_id="profile_light_roast",
        sources=[
            NodeOrEdgeData(
                source=roast_profile_view_id,
                properties={
                    "name": "New England Light Roast",
                    "description": "Light roast — high acidity, bright floral notes",
                    "tags": ["light", "high-acidity"],
                    "targetTemperatureCelsius": 182.0,
                    "roastDegree": "light",
                    "roastDurationMinutes": 10.5,
                },
            )
        ],
    ),
    NodeApply(
        space=SPACE,
        external_id="profile_medium_roast",
        sources=[
            NodeOrEdgeData(
                source=roast_profile_view_id,
                properties={
                    "name": "City Medium Roast",
                    "description": "Balanced roast — moderate acidity, caramel sweetness",
                    "tags": ["medium", "balanced"],
                    "targetTemperatureCelsius": 200.0,
                    "roastDegree": "medium",
                    "roastDurationMinutes": 13.0,
                },
            )
        ],
    ),
    NodeApply(
        space=SPACE,
        external_id="profile_dark_roast",
        sources=[
            NodeOrEdgeData(
                source=roast_profile_view_id,
                properties={
                    "name": "French Dark Roast",
                    "description": "Dark roast — heavy body, low acidity. The smoke alarm in the main notebook fired during a batch of this profile.",
                    "tags": ["dark", "french", "espresso"],
                    "targetTemperatureCelsius": 220.0,
                    "roastDegree": "french",
                    "roastDurationMinutes": 18.0,
                },
            )
        ],
    ),
]

result = client.data_modeling.instances.apply(nodes=profiles)
print(f"Created {len(result.nodes)} roast profiles:")
for n in result.nodes:
    print(f"  {n.space}/{n.external_id}  version={n.version}")

### ✅ Verify in Fusion

Search for `New England Light Roast`. You should find a node with the custom properties (`targetTemperatureCelsius`, `roastDegree`, `roastDurationMinutes`) plus the inherited CDM properties (`name`, `description`, `tags`). All three profiles should appear under the `RoastProfile` view.

If you don't see them, try refreshing the page.

## What you built

- Two custom **containers** and **views** extending CDM (`Alarm` extends `CogniteActivity`, `RoastProfile` extends `CogniteDescribable`), with `requires` constraints for query performance.
- An **enterprise/domain data model** (`CoffeeRoasteryExtension`) — the complete schema.
- A **solution data model** (`RoastQualitySolution`) — a curated subset for a specific application.
- Enriched an existing CDM node in place (same `external_id`, new container, new property).

## Cleanup

Deletes everything created in this notebook. Deleting the `Alarm` and `RoastProfile` containers removes the data stored in those containers (including `peakTemperatureCelsius` on `issue_smoke_alarm`). The `issue_smoke_alarm` node itself is not deleted — its CDM properties (name, tags, assets, etc.) live in the `cdf_cdm` containers and are unaffected.

`run_cleanup` is `False` by default, so **Run All** leaves the schema and data in place for you to explore in Fusion. Set it to `True` and run this cell when you want the extension removed.

In [ ]:
run_cleanup = False

if run_cleanup:
    # Delete RoastProfile nodes (fully owned by this notebook)
    client.data_modeling.instances.delete(
        nodes=[
            NodeId(SPACE, "profile_light_roast"),
            NodeId(SPACE, "profile_medium_roast"),
            NodeId(SPACE, "profile_dark_roast"),
        ]
    )
    print("Deleted RoastProfile nodes")

    # Delete schema objects in reverse dependency order:
    # data model → views → containers
    client.data_modeling.data_models.delete(
        [
            (SPACE, "RoastQualitySolution", "v1"),
            (SPACE, "CoffeeRoasteryExtension", "v1"),
        ]
    )
    print("Deleted data models: RoastQualitySolution, CoffeeRoasteryExtension")

    client.data_modeling.views.delete(
        [
            ViewId(SPACE, "Alarm", "v1"),
            ViewId(SPACE, "RoastProfile", "v1"),
        ]
    )
    print("Deleted views: Alarm v1, RoastProfile v1")

    # Deleting a container removes all data stored in it. This clears
    # peakTemperatureCelsius from issue_smoke_alarm without deleting the node.
    client.data_modeling.containers.delete(
        [
            ContainerId(SPACE, "Alarm"),
            ContainerId(SPACE, "RoastProfile"),
        ]
    )
    print("Deleted containers: Alarm, RoastProfile")
else:
    print("Cleanup skipped — set run_cleanup = True to execute.")